In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
data= df.drop('Order_ID', axis=1)

In [ ]:
# Task 2:
# cheaking for missing value
print(data.isnull().sum())


In [ ]:
# Task 2:
# cheaking for missing value
def check_missing_values(df):
  missing_values = data.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
data

In [ ]:
# Task 2:
## handle missing value
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Distance_km' , 'Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Delivery_Time' ]
df_clean = data[stat_cols].copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset= stat_cols)
print(f"After dropping missing : {df_clean.shape}")

In [ ]:
# Task 2:
# cheaking for missing value
def check_missing_values(df):
  missing_values = df_clean.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
## perfect we handle missing

In [ ]:
# Task 3:
# 4. Check for duplicates
def check_duplicates(df):

  duplicates = df_clean.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df_clean

In [ ]:
# Task 4:
# 3. Do we have categorical columns?
def encode_categorical_columns(df):
    categorical_cols = df_clean.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(df_clean)

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df_clean.select_dtypes(include=["number"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
df_clean

In [ ]:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

print('data before encoding:\n', categorical_cols ) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(categorical_cols ) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 5: feature scaling
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=['number']).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
# Task 6: check for imbalance
print(df_clean['Delivery_Time'].value_counts())
print(df_clean['Delivery_Time'].value_counts(normalize=True))    # it is not imbalance
data['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
# Task 1:
X = df.drop('Delivery_Time',axis=1)
y = df['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

import numpy as np

skf = KFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores.append(f1)



In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [ ]:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

  kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)


In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
import pandas as pd

feature_importance = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df['Feature'],
    importance_df['Importance']
)
plt.gca().invert_yaxis()
plt.xlabel('Importance Score')
plt.title('CatBoost Feature Importance')
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: